# NOCC 6-Block CNN — Filtering & Ablation Study (No Color Features)

This notebook trains a **6-block CNN** to predict expert color scores from **NOCC TIFF images**, and runs an ablation study across:

1. **NOCC preprocessing**: raw NOCC, κ-sigma filtering at **k = 1.5** and **k = 2.0**, and **mean inpainting** of the filtered background
2. **Architecture**: number of conv blocks (4 / 5 / 6 / 7), with 6 as the reference model
3. **Color space**: RGB / LAB / HSV / XYZ
4. **Loss & head**: single regression head + MSE, vs. dual (ordinal + regression) head with composite loss (and ordinal-only / regression-only weight extremes)

The **color-feature branch (LAB stats vector) is deliberately left out for the whole study** — every model here is CNN-image-only, matching the `M4` "no color features" configuration.

## ⚠️ Before running

- Set `DATA_ROOT` below to the actual path of the `nocc/csvs00` folder on your server (this notebook was built without access to that filesystem, so the path and the score-subfolder layout in `discover_dataset()` are **assumptions** — adjust them to match your real folder structure).
- `QUICK_MODE = True` runs short training passes for fast iteration/sanity-checking. Set it to `False` for the full 400-epoch runs once the pipeline is confirmed working on your data.
- Running the **full ablation matrix** (14 configs × 5-fold CV × up to 400 epochs) is expensive — consider running on a GPU and/or trimming `ABLATION_CONFIGS`.


## 1. Imports

In [ ]:
import os, sys, glob, random, time, json, warnings
from pathlib import Path
from typing import Optional, Dict, List

import numpy as np
import pandas as pd
import cv2
import tifffile
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import r2_score, mean_squared_error

warnings.filterwarnings("ignore")

# pip install tifffile opencv-python torch scikit-learn pandas tqdm matplotlib   # if any are missing


## 2. Reproducibility & device

In [ ]:
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## 3. Config

In [ ]:
from pathlib import Path
import numpy as np

# ---- DATA CONFIGURATION ----
DATA_ROOT = Path(
    r"F:\Aymen\server\Database_CSVS\CSVS_Zucchini_202602_LaMojonena\Zucchini_COLOR_Evaluation\Zucchini_GREEN_FOR_SPAIN_on_LVS_platform\OUTPUT\nocc\CSVS01"
)

OUTPUT_DIR = Path("./nocc_ablation_results")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

QUICK_MODE = False

IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 15 if QUICK_MODE else 400
PATIENCE = 5 if QUICK_MODE else 30
LR_PATIENCE = 3 if QUICK_MODE else 8
K_FOLDS = 5

NUM_WORKERS  = 0
# Dataset folders are Score=01 ... Score=08
IDX_TO_SCORE = np.arange(1, 9)

FG_THRESH = 10

NORM_MEAN = 0.5
NORM_STD = 0.5

TARGET_NOISE_STD = 0.15

ENSEMBLE_W_ORDINAL = 0.7
ENSEMBLE_W_REG = 0.3

USE_COLOR_FEATURES = False

print("DATA_ROOT:", DATA_ROOT)
print("Exists:", DATA_ROOT.exists())

## 4. Data discovery

Assumes one subfolder per expert score under `DATA_ROOT`, e.g. `csvs00/1/*.tiff ... csvs00/9/*.tiff`.
**Adjust `discover_dataset()` if your real layout differs** (flat folder + CSV of labels, filename-encoded score, etc.).

In [ ]:
def discover_dataset(root: Path) -> pd.DataFrame:
    records = []

    for score_dir in sorted(root.iterdir()):
        if not score_dir.is_dir():
            continue

        folder_name = score_dir.name.strip()

        # Accept folders like Score=01, Score=1, score=03, etc.
        if folder_name.lower().startswith("score="):
            try:
                score = int(folder_name.split("=")[1])
            except (IndexError, ValueError):
                continue
        else:
            continue

        if score not in IDX_TO_SCORE:
            continue

        for ext in ("*.tif", "*.tiff"):
            for p in score_dir.glob(ext):
                records.append({
                    "path": str(p),
                    "score": score
                })

    df = pd.DataFrame(records)

    if df.empty:
        raise FileNotFoundError(
            f"No TIFF images found under {root}. "
            "Expected folders like Score=01, Score=02, ..., Score=08."
        )

    return df


df_meta = None

if DATA_ROOT.exists():
    df_meta = discover_dataset(DATA_ROOT)

    print(f"Found {len(df_meta)} images across {df_meta['score'].nunique()} score classes")
    print(df_meta["score"].value_counts().sort_index())

else:
    print(
        f"DATA_ROOT {DATA_ROOT} does not exist yet -- "
        "set the correct server path in the config cell above."
    )

## 5. TIFF loading

In [ ]:
def load_tiff_16bit(path: str) -> np.ndarray:
    """Loads a (possibly 16-bit) TIFF and returns float32 RGB in [0, 1]."""
    img = tifffile.imread(path)
    if img.ndim == 2:
        img = np.stack([img] * 3, axis=-1)
    if img.shape[-1] == 4:
        img = img[..., :3]
    img = img.astype(np.float32)
    max_val = 65535.0 if img.max() > 255 else 255.0
    img = img / max_val
    return np.clip(img, 0.0, 1.0)


## 6. NOCC filtering — raw / κ-sigma (k=1.5, k=2.0) / mean inpainting

`foreground_mask_ksigma` flags pixels that are more than `k` per-channel standard deviations away from
the image's dominant (background) color as foreground. `apply_ksigma_filter` zeroes the background;
`mean_inpaint` instead fills it with the mean foreground color, avoiding a hard black background.

In [ ]:
def foreground_mask_ksigma(img_rgb: np.ndarray, k: float, fg_thresh: float = FG_THRESH) -> np.ndarray:
    flat = img_rgb.reshape(-1, 3)
    mean = flat.mean(axis=0)
    std = flat.std(axis=0) + 1e-8
    dist = np.abs(img_rgb - mean) / std      # per-channel z-score
    dist = dist.max(axis=-1)                  # worst-case channel
    mask = dist > k                            # True = far from background -> foreground

    brightness = img_rgb.mean(axis=-1) * 255.0
    mask &= (brightness > fg_thresh) & (brightness < 255 - fg_thresh)
    return mask


def apply_ksigma_filter(img_rgb: np.ndarray, k: float) -> np.ndarray:
    """Zeros out background pixels identified by the k-sigma mask."""
    mask = foreground_mask_ksigma(img_rgb, k)
    out = img_rgb.copy()
    out[~mask] = 0.0
    return out


def mean_inpaint(img_rgb: np.ndarray, k: float) -> np.ndarray:
    """k-sigma background mask, filled with the mean foreground color instead of zeros."""
    mask = foreground_mask_ksigma(img_rgb, k)
    out = img_rgb.copy()
    fg_mean = img_rgb[mask].mean(axis=0) if mask.any() else img_rgb.reshape(-1, 3).mean(axis=0)
    out[~mask] = fg_mean
    return out


def preprocess_nocc(img_rgb: np.ndarray, method: str, k: Optional[float] = None) -> np.ndarray:
    """method: 'raw' | 'ksigma' | 'mean_inpaint'"""
    if method == "raw":
        return img_rgb
    elif method == "ksigma":
        return apply_ksigma_filter(img_rgb, k)
    elif method == "mean_inpaint":
        return mean_inpaint(img_rgb, k)
    else:
        raise ValueError(f"Unknown preprocessing method: {method}")


### 6b. Visual sanity check (run once real data is available)

In [ ]:
if df_meta is not None:
    sample_path = df_meta.iloc[0]["path"]
    raw = load_tiff_16bit(sample_path)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (title, im) in zip(axes, [
        ("raw NOCC", raw),
        ("k-sigma filter, k=1.5", apply_ksigma_filter(raw, 1.5)),
        ("k-sigma filter, k=2.0", apply_ksigma_filter(raw, 2.0)),
        ("mean inpaint, k=2.0", mean_inpaint(raw, 2.0)),
    ]):
        ax.imshow(im)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


## 7. Color-space conversion

In [ ]:
def to_color_space(img_rgb_uint8: np.ndarray, space: str) -> np.ndarray:
    """img_rgb_uint8: HxWx3 uint8 RGB. Returns float32, roughly scaled to [0, 1]."""
    if space == "RGB":
        return img_rgb_uint8.astype(np.float32) / 255.0
    elif space == "LAB":
        lab = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2LAB).astype(np.float32)
        lab[..., 0] /= 100.0
        lab[..., 1] = (lab[..., 1] + 128.0) / 255.0
        lab[..., 2] = (lab[..., 2] + 128.0) / 255.0
        return lab
    elif space == "HSV":
        hsv = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2HSV).astype(np.float32)
        hsv[..., 0] /= 179.0
        hsv[..., 1] /= 255.0
        hsv[..., 2] /= 255.0
        return hsv
    elif space == "XYZ":
        xyz = cv2.cvtColor(img_rgb_uint8, cv2.COLOR_RGB2XYZ).astype(np.float32)
        return xyz / 255.0
    else:
        raise ValueError(f"Unknown color space: {space}")


## 8. Dataset

In [ ]:
class NOCCDataset(Dataset):
    """
    filter_method: 'raw' | 'ksigma' | 'mean_inpaint'   (NOCC preprocessing ablation)
    color_space:   'RGB' | 'LAB' | 'HSV' | 'XYZ'      (color-space ablation)
    No color-feature vector is ever returned -- this study is CNN-image-only.
    """
    def __init__(self, df: pd.DataFrame, filter_method: str = "raw", k: Optional[float] = None,
                 color_space: str = "RGB", img_size: int = IMG_SIZE, augment: bool = False,
                 label_noise_std: float = 0.0):
        self.df = df.reset_index(drop=True)
        self.filter_method = filter_method
        self.k = k
        self.color_space = color_space
        self.img_size = img_size
        self.augment = augment
        self.label_noise_std = label_noise_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = load_tiff_16bit(row["path"])                       # float32 [0,1] RGB
        img = preprocess_nocc(img, self.filter_method, self.k)   # NOCC filtering ablation
        img_uint8 = (img * 255).astype(np.uint8)
        img = to_color_space(img_uint8, self.color_space)        # color-space ablation
        img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)

        if self.augment:
            if random.random() < 0.5:
                img = np.fliplr(img).copy()
            if random.random() < 0.5:
                img = np.flipud(img).copy()

        img = (img - NORM_MEAN) / NORM_STD
        img_t = torch.from_numpy(img.transpose(2, 0, 1).astype(np.float32))

        score = float(row["score"])
        if self.augment and self.label_noise_std > 0:
            score = score + np.random.normal(0, self.label_noise_std)

        ordinal_target = torch.zeros(len(IDX_TO_SCORE) - 1, dtype=torch.float32)
        ordinal_target[: int(row["score"]) - 1] = 1.0             # cumulative ordinal encoding

        return {
            "image": img_t,
            "score": torch.tensor(score, dtype=torch.float32),
            "ordinal_target": ordinal_target,
        }


## 9. 6-block CNN backbone (block count is the architecture-ablation knob)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ELU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ELU(inplace=True),
        )
        self.pool = nn.MaxPool2d(2) if pool else nn.Identity()

    def forward(self, x):
        return self.pool(self.conv(x))


class CNNBackbone(nn.Module):
    """Depth-doubling conv backbone. n_blocks defaults to 6 (the reference architecture)."""
    def __init__(self, in_ch=3, base_ch=16, n_blocks=6):
        super().__init__()
        chans = [in_ch] + [base_ch * (2 ** i) for i in range(n_blocks)]
        self.blocks = nn.ModuleList([ConvBlock(chans[i], chans[i + 1]) for i in range(n_blocks)])
        self.out_ch = chans[-1]
        self.gap = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        return self.gap(x).flatten(1)


## 10. Heads (single regression vs. dual ordinal+regression)

In [ ]:
class RegressionHead(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(in_dim, 64), nn.ELU(), nn.Linear(64, 1))

    def forward(self, x):
        return self.fc(x).squeeze(-1)


class DualHead(nn.Module):
    """Ordinal head (n_classes-1 cumulative logits) + regression head."""
    def __init__(self, in_dim, n_classes=len(IDX_TO_SCORE)):
        super().__init__()
        self.reg = nn.Sequential(nn.Linear(in_dim, 64), nn.ELU(), nn.Linear(64, 1))
        self.ord = nn.Sequential(nn.Linear(in_dim, 64), nn.ELU(), nn.Linear(64, n_classes - 1))

    def forward(self, x):
        return {"reg": self.reg(x).squeeze(-1), "ord": self.ord(x)}


## 11. Full model — no color-feature branch

In [ ]:
class NOCCModel(nn.Module):
    """
    Configurable-depth CNN (6 blocks by default) with either a single regression head
    or a dual ordinal+regression head. No color-feature vector is concatenated anywhere
    in this study (use_color_features is fixed False) -- the model relies purely on the
    CNN's own learned image features.
    """
    def __init__(self, n_blocks=6, base_ch=16, head_type="dual", in_ch=3):
        super().__init__()
        self.backbone = CNNBackbone(in_ch=in_ch, base_ch=base_ch, n_blocks=n_blocks)
        self.head_type = head_type
        if head_type == "single":
            self.head = RegressionHead(self.backbone.out_ch)
        elif head_type == "dual":
            self.head = DualHead(self.backbone.out_ch)
        else:
            raise ValueError(head_type)

    def forward(self, x):
        return self.head(self.backbone(x))


## 12. Losses

In [ ]:
def ordinal_to_score(ord_probs: torch.Tensor) -> torch.Tensor:
    return ord_probs.sum(dim=1) + 1.0


def composite_loss(outputs, batch, w_ord=0.5, w_reg=0.5):
    reg_loss = F.mse_loss(outputs["reg"], batch["score"])
    ord_loss = F.binary_cross_entropy_with_logits(outputs["ord"], batch["ordinal_target"])
    total = w_reg * reg_loss + w_ord * ord_loss
    return total


def mse_only_loss(outputs, batch):
    pred = outputs if torch.is_tensor(outputs) else outputs["reg"]
    return F.mse_loss(pred, batch["score"])


## 13. Metrics

In [ ]:
def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    acc0 = float(np.mean(np.round(y_pred) == y_true))
    acc1 = float(np.mean(np.abs(np.round(y_pred) - y_true) <= 1))
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    return {"Acc@0": acc0, "Acc@1": acc1, "R2": r2, "RMSE": rmse}


## 14. Train / eval loop for one fold

In [ ]:
def predict_score(model, outputs, w_ordinal=ENSEMBLE_W_ORDINAL, w_reg=ENSEMBLE_W_REG):
    if model.head_type == "single":
        return outputs
    ord_probs = torch.sigmoid(outputs["ord"])
    ord_score = ordinal_to_score(ord_probs)
    return w_ordinal * ord_score + w_reg * outputs["reg"]


def compute_loss(model, outputs, batch, config):
    if config["loss_type"] == "mse":
        return mse_only_loss(outputs, batch)
    return composite_loss(outputs, batch, **config.get("loss_weights", {"w_ord": 0.5, "w_reg": 0.5}))


def run_one_fold(train_df, val_df, config, fold_idx):
    seed_everything(SEED + fold_idx)

    train_ds = NOCCDataset(train_df, config["filter_method"], config.get("k"), config["color_space"],
                            augment=True, label_noise_std=TARGET_NOISE_STD)
    val_ds = NOCCDataset(val_df, config["filter_method"], config.get("k"), config["color_space"],
                          augment=False)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = NOCCModel(n_blocks=config.get("n_blocks", 6), head_type=config["head_type"]).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=LR_PATIENCE)

    best_val_loss, best_state, epochs_no_improve = float("inf"), None, 0

    for epoch in range(EPOCHS):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(batch["image"])
            loss = compute_loss(model, outputs, batch, config)
            loss.backward()
            optimizer.step()

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                outputs = model(batch["image"])
                val_losses.append(compute_loss(model, outputs, batch, config).item())
        val_loss = float(np.mean(val_losses))
        scheduler.step(val_loss)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(batch["image"])
            pred = predict_score(model, outputs)
            y_true.extend(batch["score"].cpu().numpy().tolist())
            y_pred.extend(pred.cpu().numpy().tolist())

    return compute_metrics(y_true, y_pred)


## 15. 5-fold cross-validated experiment runner

In [ ]:
def run_experiment(df, config, k_folds=K_FOLDS):
    skf = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=SEED)
    fold_metrics = []
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(df, df["score"])):
        train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]
        metrics = run_one_fold(train_df, val_df, config, fold_idx)
        metrics["fold"] = fold_idx
        fold_metrics.append(metrics)
        print(f"  [{config['name']}] fold {fold_idx}: " +
              ", ".join(f"{k}={v:.4f}" for k, v in metrics.items() if k != "fold"))
    fold_df = pd.DataFrame(fold_metrics)
    summary = fold_df.drop(columns="fold").agg(["mean", "std"]).to_dict()
    return fold_df, summary


## 16. Ablation matrix

Baseline = 6-block backbone, mean-inpainted NOCC at k=2.0, LAB color space, dual head, composite loss (50/50), **no color features**.
Every other config changes exactly one axis at a time relative to this baseline.

In [ ]:
BASELINE = {
    "name": "baseline_6block_meaninpaint_k2_LAB_dual_composite",
    "filter_method": "mean_inpaint",
    "k": 2.0,
    "color_space": "LAB",
    "n_blocks": 6,
    "head_type": "dual",
    "loss_type": "composite",
    "loss_weights": {"w_ord": 0.5, "w_reg": 0.5},
}

def variant(name, **overrides):
    cfg = {**BASELINE, **overrides, "name": name}
    return cfg

ABLATION_CONFIGS = [
    BASELINE,

    # --- NOCC filtering / inpainting ablation ---
    variant("filter_raw",                filter_method="raw",          k=None),
    variant("filter_ksigma_k1.5",        filter_method="ksigma",       k=1.5),
    variant("filter_ksigma_k2.0",        filter_method="ksigma",       k=2.0),
    variant("filter_mean_inpaint_k1.5",  filter_method="mean_inpaint", k=1.5),
    # baseline already covers mean_inpaint @ k=2.0

    # --- Architecture ablation (block count) ---
    variant("arch_4blocks", n_blocks=4),
    variant("arch_5blocks", n_blocks=5),
    variant("arch_7blocks", n_blocks=7),

    # --- Color-space ablation ---
    variant("colorspace_RGB", color_space="RGB"),
    variant("colorspace_HSV", color_space="HSV"),
    variant("colorspace_XYZ", color_space="XYZ"),

    # --- Loss / head ablation ---
    variant("head_single_loss_mse", head_type="single", loss_type="mse"),
    variant("head_dual_loss_ordinal_only", loss_type="composite",
            loss_weights={"w_ord": 1.0, "w_reg": 0.0}),
    variant("head_dual_loss_reg_only", loss_type="composite",
            loss_weights={"w_ord": 0.0, "w_reg": 1.0}),
]

for c in ABLATION_CONFIGS:
    c["use_color_features"] = False   # enforced across the whole study

print(f"{len(ABLATION_CONFIGS)} configs queued")
for c in ABLATION_CONFIGS:
    print(" -", c["name"])


## 17. Run all experiments

In [ ]:
all_results = {}

if df_meta is not None:
    for config in ABLATION_CONFIGS:
        print(f"\n=== Running: {config['name']} ===")
        fold_df, summary = run_experiment(df_meta, config)
        all_results[config["name"]] = {"folds": fold_df, "summary": summary}
        fold_df.to_csv(OUTPUT_DIR / f"{config['name']}_folds.csv", index=False)

    with open(OUTPUT_DIR / "ablation_summary.json", "w") as f:
        json.dump({k: v["summary"] for k, v in all_results.items()}, f, indent=2, default=str)
else:
    print("Skipping: set DATA_ROOT and re-run from section 4 first.")


## 18. Aggregate results table

In [ ]:
rows = []
for name, res in all_results.items():
    s = res["summary"]
    rows.append({
        "config": name,
        "R2_mean": s["R2"]["mean"], "R2_std": s["R2"]["std"],
        "RMSE_mean": s["RMSE"]["mean"], "RMSE_std": s["RMSE"]["std"],
        "Acc@0_mean": s["Acc@0"]["mean"], "Acc@1_mean": s["Acc@1"]["mean"],
    })

results_table = pd.DataFrame(rows).sort_values("R2_mean", ascending=False).reset_index(drop=True)
if not results_table.empty:
    results_table.to_csv(OUTPUT_DIR / "ablation_results_table.csv", index=False)
results_table


## 19. Plots per ablation axis

In [ ]:
def plot_group(table, names, title):
    sub = table[table["config"].isin(names)]
    if sub.empty:
        return
    plt.figure(figsize=(8, 4))
    plt.bar(sub["config"], sub["R2_mean"], yerr=sub["R2_std"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("R\u00b2 (mean \u00b1 std, 5-fold CV)")
    plt.title(title)
    plt.tight_layout()
    plt.show()

if not results_table.empty:
    plot_group(results_table,
               [BASELINE["name"], "filter_raw", "filter_ksigma_k1.5", "filter_ksigma_k2.0", "filter_mean_inpaint_k1.5"],
               "NOCC filtering / inpainting ablation")

    plot_group(results_table,
               [BASELINE["name"], "arch_4blocks", "arch_5blocks", "arch_7blocks"],
               "Architecture depth ablation")

    plot_group(results_table,
               [BASELINE["name"], "colorspace_RGB", "colorspace_HSV", "colorspace_XYZ"],
               "Color-space ablation")

    plot_group(results_table,
               [BASELINE["name"], "head_single_loss_mse", "head_dual_loss_ordinal_only", "head_dual_loss_reg_only"],
               "Loss / head ablation")
